# 1. Data Preprocessing
## Banking Customer Churn Analytics

### Dataset Schema Overview
- **Unnecessary Columns**: `RowNumber`, `CustomerId`, `Surname`
- **Demographics**: `Geography` (France, Germany, Spain), `Gender`, `Age`
- **Financial Attributes**: `Tenure`, `Balance`, `NumOfProducts`, `HasCrCard`, `IsActiveMember`, `EstimatedSalary`
- **Target Variable**: `Exited` (1 = Churned, 0 = Retained)


In [1]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath(".."))
from src.data_loader import BankDataLoader
from src.eda import BankChurnEDA

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)


# 1. Load & Validate Raw Banking Data

In [2]:
# Load raw dataset
data_path = os.path.abspath("../data/raw/bank_customer_churn_raw.csv")
loader = BankDataLoader(data_path)
df = loader.load_or_create()

print(f"Dataset Shape: {df.shape}")
df.head()


[DataLoader] Loading customer dataset from: C:\Users\asus\.gemini\antigravity-ide\scratch\banking-churn-analytics\data\raw\bank_customer_churn_raw.csv
[DataLoader] Schema validation passed. Records: 10,000, Columns: 14
Dataset Shape: (10000, 14)


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


## 2. Data Health & Quality Profiling

In [3]:
eda_engine = BankChurnEDA(df)
audit_results = eda_engine.perform_quality_audit()

print(f"Total Audited Records: {audit_results['total_records']:,}")
print(f"Duplicate Customer IDs: {audit_results['duplicate_customer_ids']}")
print(f"Overall Churn Rate:     {audit_results['churn_rate']:.2%}")
print("\nMissing Value Counts by Column:")
print(pd.Series(audit_results['missing_values']))


Total Audited Records: 10,000
Duplicate Customer IDs: 0
Overall Churn Rate:     20.37%

Missing Value Counts by Column:
RowNumber          0
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64


## 3. Summary Statistics & Data Types

In [4]:
# Numerical features descriptive statistics
df.describe().T.round(2)


,count,mean,std,min,25%,50%,75%,max
RowNumber,10000.0,5000.50,2886.90,1.00,2500.75,5000.50,7500.25,10000.00
CustomerId,10000.0,15690940.57,71936.19,15565701.00,15628528.25,15690738.00,15753233.75,15815690.00
CreditScore,10000.0,650.53,96.65,350.00,584.00,652.00,718.00,850.00
Age,10000.0,38.92,10.49,18.00,32.00,37.00,44.00,92.00
Tenure,10000.0,5.01,2.89,0.00,3.00,5.00,7.00,10.00
Balance,10000.0,76485.89,62397.41,0.00,0.00,97198.54,127644.24,250898.09
NumOfProducts,10000.0,1.53,0.58,1.00,1.00,1.00,2.00,4.00
HasCrCard,10000.0,0.71,0.46,0.00,0.00,1.00,1.00,1.00
IsActiveMember,10000.0,0.52,0.50,0.00,0.00,1.00,1.00,1.00
EstimatedSalary,10000.0,100090.24,57510.49,11.58,51002.11,100193.92,149388.25,199992.48


## 4. Outlier Detection (Interquartile Range - IQR)

In [5]:
outliers_df = pd.DataFrame(audit_results['outliers_iqr']).T
outliers_df


,outlier_count,outlier_pct,lower_bound,upper_bound
CreditScore,15.0,0.15,383.00,919.00
Age,359.0,3.59,14.00,62.00
Tenure,0.0,0.00,-3.00,13.00
Balance,0.0,0.00,-191466.36,319110.60
NumOfProducts,60.0,0.60,-0.50,3.50
HasCrCard,0.0,0.00,-1.50,2.50
IsActiveMember,0.0,0.00,-1.50,2.50
EstimatedSalary,0.0,0.00,-96577.10,296967.45


## 5. Export Cleaned Dataset for Downstream Analysis

In [6]:
processed_path = os.path.abspath("../data/processed/bank_customer_churn_clean.csv")
os.makedirs(os.path.dirname(processed_path), exist_ok=True)
df.to_csv(processed_path, index=False)
print(f"Successfully saved clean dataset ")


Successfully saved clean dataset to: C:\Users\asus\.gemini\antigravity-ide\scratch\banking-churn-analytics\data\processed\bank_customer_churn_clean.csv
